# Задание 4. Сравнение алгоритмов регрессии

Цель: Исследовать и сравнить эффективность алгоритмов машинного обучения для задачи регрессии: SVR, DT, LR и RF на датасете diabetes.

In [1]:
# 1. Импорт библиотек и загрузка данных
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных diabetes
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

print(f"Датасет: Diabetes")
print(f"Размер: {X.shape}")
print(f"Описание: Прогнозирование прогрессии диабета через год")
print(f"\nПризнаки: {diabetes.feature_names}")

Датасет: Diabetes
Размер: (442, 10)
Описание: Прогнозирование прогрессии диабета через год

Признаки: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [2]:
# Разделение на обучающую и тестовую выборки (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (309, 10)
Размер тестовой выборки: (133, 10)


## 2. Создание конвейеров для каждого алгоритма

In [3]:
# Создание конвейеров
svr_pipeline = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
dt_pipeline = Pipeline([('scaler', StandardScaler()), ('dt', DecisionTreeRegressor(random_state=42))])
lr_pipeline = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
rf_pipeline = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor(random_state=42))])

print("Конвейеры созданы для всех алгоритмов")

Конвейеры созданы для всех алгоритмов


## 3. Настройка гиперпараметров

In [4]:
# Параметры для GridSearchCV
svr_params = {
    'svr__C': [0.1, 1, 10],
    'svr__kernel': ['rbf', 'linear'],
    'svr__epsilon': [0.01, 0.1, 0.5]
}

dt_params = {
    'dt__max_depth': [None, 3, 5, 7],
    'dt__min_samples_split': [2, 5, 10],
    'dt__min_samples_leaf': [1, 2, 4]
}

lr_params = {
    'lr__fit_intercept': [True, False]
}

rf_params = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 5, 10],
    'rf__min_samples_split': [2, 5, 10]
}

print("Параметры для подбора определены")

Параметры для подбора определены


## 4. Обучение и оценка моделей

In [5]:
# Обучение SVR
print("Обучение SVR...")
svr_grid = GridSearchCV(svr_pipeline, svr_params, cv=5, scoring='r2', n_jobs=-1)
svr_grid.fit(X_train, y_train)
svr_pred = svr_grid.predict(X_test)

svr_mse = mean_squared_error(y_test, svr_pred)
svr_mae = mean_absolute_error(y_test, svr_pred)
svr_r2 = r2_score(y_test, svr_pred)

print(f"SVR - Лучшие параметры: {svr_grid.best_params_}")
print(f"SVR - MSE: {svr_mse:.2f}, MAE: {svr_mae:.2f}, R²: {svr_r2:.4f}\n")

Обучение SVR...
SVR - Лучшие параметры: {'svr__C': 10, 'svr__epsilon': 0.5, 'svr__kernel': 'linear'}
SVR - MSE: 2910.73, MAE: 42.53, R²: 0.4608



In [6]:
# Обучение Decision Tree
print("Обучение Decision Tree...")
dt_grid = GridSearchCV(dt_pipeline, dt_params, cv=5, scoring='r2', n_jobs=-1)
dt_grid.fit(X_train, y_train)
dt_pred = dt_grid.predict(X_test)

dt_mse = mean_squared_error(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_r2 = r2_score(y_test, dt_pred)

print(f"DT - Лучшие параметры: {dt_grid.best_params_}")
print(f"DT - MSE: {dt_mse:.2f}, MAE: {dt_mae:.2f}, R²: {dt_r2:.4f}\n")

Обучение Decision Tree...
DT - Лучшие параметры: {'dt__max_depth': 3, 'dt__min_samples_leaf': 2, 'dt__min_samples_split': 2}
DT - MSE: 3616.77, MAE: 46.96, R²: 0.3300



In [7]:
# Обучение Linear Regression
print("Обучение Linear Regression...")
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring='r2', n_jobs=-1)
lr_grid.fit(X_train, y_train)
lr_pred = lr_grid.predict(X_test)

lr_mse = mean_squared_error(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_r2 = r2_score(y_test, lr_pred)

print(f"LR - Лучшие параметры: {lr_grid.best_params_}")
print(f"LR - MSE: {lr_mse:.2f}, MAE: {lr_mae:.2f}, R²: {lr_r2:.4f}\n")

Обучение Linear Regression...
LR - Лучшие параметры: {'lr__fit_intercept': True}
LR - MSE: 2821.75, MAE: 41.92, R²: 0.4773



In [8]:
# Обучение Random Forest
print("Обучение Random Forest...")
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring='r2', n_jobs=-1)
rf_grid.fit(X_train, y_train)
rf_pred = rf_grid.predict(X_test)

rf_mse = mean_squared_error(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print(f"RF - Лучшие параметры: {rf_grid.best_params_}")
print(f"RF - MSE: {rf_mse:.2f}, MAE: {rf_mae:.2f}, R²: {rf_r2:.4f}\n")

Обучение Random Forest...
RF - Лучшие параметры: {'rf__max_depth': None, 'rf__min_samples_split': 10, 'rf__n_estimators': 100}
RF - MSE: 2788.02, MAE: 42.28, R²: 0.4835



In [9]:
# Сравнение результатов
results = pd.DataFrame({
    'Алгоритм': ['SVR', 'Decision Tree', 'Linear Regression', 'Random Forest'],
    'MSE': [svr_mse, dt_mse, lr_mse, rf_mse],
    'MAE': [svr_mae, dt_mae, lr_mae, rf_mae],
    'R²': [svr_r2, dt_r2, lr_r2, rf_r2],
    'Лучшие параметры': [
        str(svr_grid.best_params_),
        str(dt_grid.best_params_),
        str(lr_grid.best_params_),
        str(rf_grid.best_params_)
    ]
})

results = results.sort_values('R²', ascending=False)

print("\n" + "="*120)
print("СРАВНЕНИЕ АЛГОРИТМОВ РЕГРЕССИИ")
print("="*120)
print(results.to_string(index=False))
print("="*120)
print("\nПримечание: Лучшая модель имеет наибольший R² и наименьшие MSE/MAE")


СРАВНЕНИЕ АЛГОРИТМОВ РЕГРЕССИИ
         Алгоритм         MSE       MAE       R²                                                              Лучшие параметры
    Random Forest 2788.017115 42.282669 0.483539 {'rf__max_depth': None, 'rf__min_samples_split': 10, 'rf__n_estimators': 100}
Linear Regression 2821.750981 41.919378 0.477290                                                   {'lr__fit_intercept': True}
              SVR 2910.731438 42.529921 0.460807                  {'svr__C': 10, 'svr__epsilon': 0.5, 'svr__kernel': 'linear'}
    Decision Tree 3616.769895 46.957630 0.330018   {'dt__max_depth': 3, 'dt__min_samples_leaf': 2, 'dt__min_samples_split': 2}

Примечание: Лучшая модель имеет наибольший R² и наименьшие MSE/MAE


## 5. Сохранение лучшей модели

In [10]:
# Находим лучшую модель по R²
models = {
    'SVR': (svr_grid, svr_r2),
    'DT': (dt_grid, dt_r2),
    'LR': (lr_grid, lr_r2),
    'RF': (rf_grid, rf_r2)
}

best_model_name = max(models.items(), key=lambda x: x[1][1])[0]
best_model = models[best_model_name][0]
best_r2 = models[best_model_name][1]

print(f"Лучшая модель: {best_model_name} с R² = {best_r2:.4f}")

# Сохранение модели
joblib.dump(best_model, 'best_regressor.joblib')
print("Модель сохранена в файл 'best_regressor.joblib'")

Лучшая модель: RF с R² = 0.4835
Модель сохранена в файл 'best_regressor.joblib'


## 6. Проверка импорта модели

In [11]:
# Загрузка модели
loaded_model = joblib.load('best_regressor.joblib')
print("Модель успешно загружена")

# Проверка на новых данных (первые 5 образцов тестовой выборки)
test_samples = X_test[:5]
loaded_predictions = loaded_model.predict(test_samples)
original_predictions = best_model.predict(test_samples)

comparison = pd.DataFrame({
    'Истинное значение': y_test[:5],
    'Предсказание (загруженная)': loaded_predictions,
    'Предсказание (оригинальная)': original_predictions,
    'Совпадают': np.isclose(loaded_predictions, original_predictions)
})

print("\nСравнение предсказаний:")
print(comparison.to_string(index=False))

# Метрики на всей тестовой выборке
loaded_pred_full = loaded_model.predict(X_test)
loaded_mse = mean_squared_error(y_test, loaded_pred_full)
loaded_mae = mean_absolute_error(y_test, loaded_pred_full)
loaded_r2 = r2_score(y_test, loaded_pred_full)

print(f"\nМетрики загруженной модели на тесте:")
print(f"MSE: {loaded_mse:.2f}")
print(f"MAE: {loaded_mae:.2f}")
print(f"R²: {loaded_r2:.4f}")

Модель успешно загружена

Сравнение предсказаний:
 Истинное значение  Предсказание (загруженная)  Предсказание (оригинальная)  Совпадают
             219.0                  155.831315                   155.831315       True
              70.0                  176.042203                   176.042203       True
             202.0                  139.721296                   139.721296       True
             230.0                  250.992428                   250.992428       True
             111.0                  120.492101                   120.492101       True

Метрики загруженной модели на тесте:
MSE: 2788.02
MAE: 42.28
R²: 0.4835


## 7. Выводы

In [12]:
print("\n📊 ВЫВОДЫ:")
print("="*70)
print(f"1. Лучший алгоритм: {best_model_name} с R² = {best_r2:.4f}")
print(f"2. Датасет: Diabetes - прогнозирование прогрессии диабета")
print(f"3. Модель успешно сохранена и загружена обратно")
print(f"4. Загруженная модель работает корректно")

# Обоснование выбора
print(f"\n💡 Обоснование выбора модели {best_model_name}:")
if best_model_name == 'RF':
    print("- Случайный лес показывает высокую устойчивость к переобучению")
    print("- Хорошо работает с нелинейными зависимостями")
    print("- Автоматически учитывает важность признаков")
elif best_model_name == 'LR':
    print("- Линейная регрессия проста и интерпретируема")
    print("- Хорошо работает при линейных зависимостях")
    print("- Быстрое обучение и предсказание")
elif best_model_name == 'SVR':
    print("- SVR эффективен для небольших датасетов")
    print("- Хорошо обобщает на новых данных")
    print("- Работает с нелинейными зависимостями через ядра")
else:
    print("- Дерево решений просто в интерпретации")
    print("- Не требует масштабирования признаков")
    
print("="*70)


📊 ВЫВОДЫ:
1. Лучший алгоритм: RF с R² = 0.4835
2. Датасет: Diabetes - прогнозирование прогрессии диабета
3. Модель успешно сохранена и загружена обратно
4. Загруженная модель работает корректно

💡 Обоснование выбора модели RF:
- Случайный лес показывает высокую устойчивость к переобучению
- Хорошо работает с нелинейными зависимостями
- Автоматически учитывает важность признаков
